# One grouping, two views

**Which groups are supported mainly by QC rather than expression?**

560 synthetic cells: M0–M3 have distinct molecular programs; Q0–Q2 share a program but differ in QC. QC-like covariates are simulated independently of expression—not calculated from it. This controlled example teaches the workflow, not biological validation.

New to ITCA? [Start with the concept](00_concepts.ipynb).

In [ ]:
from IPython.display import display
import scanpy as sc
import gauge_sc as gauge

adata = gauge.datasets.audit_example()
palette = dict(zip(adata.obs["group"].cat.categories, sc.pl.palettes.vega_20_scanpy))

## 1. Evaluate the same groups from two views

The molecular view describes expression; the QC view describes measurement properties. Each group's contribution combines its label information with how well it can be predicted.

In [ ]:
result = gauge.evaluate(
    adata,
    groupby="group",
    views={
        "molecular": {"obsm": "X_pca"},
        "qc": {
            "obs": ["total_counts", "n_genes_by_counts", "pct_counts_mt"],
            "log1p": ["total_counts", "n_genes_by_counts"],
        },
    },
)

## 2. Screen for QC-associated groups

Left: contributions from each view. Right: median-centered contrast. Positive contrast alone is not a candidate call; we explicitly enable descriptive screening. **The default example selects Q0–Q2: screening is not ground truth.** Candidates in real data are not automatically artifacts or doublets.

In [ ]:
audit = gauge.audit(
    result, reference="molecular", contrast="qc", maximum_mirror_ratio=1 / 3
)
fig = gauge.pl.audit(audit, palette=palette)
display(audit.table.round(3))


## 3. Locate the groups with Scanpy

Group colors stay consistent. The continuous map shows group contrast broadcast to cells—not a separate score for each cell.

In [ ]:
sc.pp.neighbors(adata, use_rep="X_pca", random_state=0)
sc.tl.umap(adata, random_state=0)
audit.write_to_adata(adata)
sc.pl.umap(adata, color="group", palette=palette)
sc.pl.umap(adata, color="gauge_sc_audit_contrast", cmap="RdBu_r", vcenter=0)

## Try your data

Replace the example with `sc.read_h5ad(...)`, choose your grouping column and declare its views.

[API and parameters](../docs/api.md) · [Screening, styling and saving](../docs/recipes.md)

[Optional: screening sensitivity](02_audit_sensitivity.ipynb)


## Work with your agent

[Install the skill and choose an interface](../docs/agents.md). Then try:

> Use $gauge to audit the grouping in this example using molecular and QC views. Explain the raw contributions and centered contrasts. Decide whether screening is useful for my question and justify its settings; show which groups need follow-up without turning the scores into cell-level calls.

These tutorial settings are an example; adapt them to your question and data.